In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [15]:
play_tennis = pd.DataFrame({
    'outlook': ['sunny', 'sunny', 'overcast', 'rain', 'rain', 'rain', 'overcast', 'sunny', 'sunny', 'rain', 'sunny', 'overcast', 'overcast', 'rain'],
    'temp': [85, 80, 83, 70, 68, 65, 64, 69, 75, 71, 81, 81, 72, 69],
    'humidity': [85, 90, 78, 96, 80, 70, 65, 70, 80, 80, 75, 88, 69, 70],
    'wind': [False, True, False, False, False, True, True, False, False, True, True, True, False, True],
    'play': ['no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'yes', 'yes', 'no']
})

In [16]:
play_tennis[['temp', 'humidity']] = play_tennis[['temp', 'humidity']].astype('category')
play_tennis.dtypes

outlook       object
temp        category
humidity    category
wind            bool
play          object
dtype: object

In [17]:
play_tennis['temp'] = play_tennis['temp'].map(
    lambda x: 'hot' if x >= 80 else 'mild' if x >= 70 else 'cool'
).astype('category')

play_tennis['humidity'] = play_tennis['humidity'].map(
    lambda x: 'high' if x >= 80 else 'normal'
).astype('category')

play_tennis[['temp', 'humidity']].head(14)

,temp,humidity
0,hot,high
1,hot,high
2,hot,normal
3,mild,high
4,cool,high
5,cool,normal
6,cool,normal
7,cool,normal
8,mild,high
9,mild,high


In [18]:
play_tennis['play'].value_counts()

play
yes    9
no     5
Name: count, dtype: int64

In [19]:
py = 9 / 14
pn = 5 / 14

In [20]:
# outlook
pd.crosstab(play_tennis['outlook'], play_tennis['play'])


play,no,yes
outlook,,
overcast,0,4
rain,3,2
sunny,2,3


In [21]:
pon = 0
prn = 3 / 5
psn = 2 / 5

poy = 4 / 9
pry = 3 / 9
psy = 2 / 9

In [22]:
pd.crosstab(play_tennis['temp'], play_tennis['play'])

play,no,yes
temp,,
cool,2,3
hot,2,3
mild,1,3


In [23]:
# Naive Bayes Prediction Example
# Predict: Outlook=Sunny, Temp=Cool, Humidity=High, Wind=TRUE

pCoolNo = 2 / 5
pCoolYes = 3 / 9

# Humidity crosstab
pd.crosstab(play_tennis['humidity'], play_tennis['play'])

play,no,yes
humidity,,
high,3,4
normal,2,5


In [24]:
# P(Humidity=High | No) = 4/5
# P(Humidity=High | Yes) = 3/9

pHighNo = 4 / 5
pHighYes = 3 / 9

# Wind crosstab
pd.crosstab(play_tennis['wind'], play_tennis['play'])

play,no,yes
wind,,
False,1,6
True,4,3


In [25]:
# P(Wind=TRUE | No) = 3/5
# P(Wind=TRUE | Yes) = 3/9

pWindNo = 3 / 5
pWndYes = 3 / 9

# Calculate posterior probabilities
# P(No | Sunny, Cool, High, TRUE) ∝ P(No) × P(Sunny|No) × P(Cool|No) × P(High|No) × P(Wind=TRUE|No)
# P(Yes | Sunny, Cool, High, TRUE) ∝ P(Yes) × P(Sunny|Yes) × P(Cool|Yes) × P(High|Yes) × P(Wind=TRUE|Yes)

In [27]:
# Posterior for No
posterior_no = pn * psn * pCoolNo * pHighNo * pWindNo
print(f"Posterior for No: {posterior_no:.4f}")

# Posterior for Yes
posterior_yes = py * psy * pCoolYes * pHighYes * pWndYes
print(f"Posterior for Yes: {posterior_yes:.4f}")

Posterior for No: 0.0274
Posterior for Yes: 0.0053


In [28]:
# Prediction
if posterior_yes > posterior_no:
    print("Prediction: YES (Play Tennis)")
else:
    print("Prediction: NO (Don't Play Tennis)")

Prediction: NO (Don't Play Tennis)


In [29]:
# Using sklearn's Naive Bayes for verification
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import LabelEncoder

# Prepare data
le_outlook = LabelEncoder()
le_temp = LabelEncoder()
le_humidity = LabelEncoder()
le_wind = LabelEncoder()
le_play = LabelEncoder()

X = pd.DataFrame({
    'outlook': le_outlook.fit_transform(play_tennis['outlook']),
    'temp': le_temp.fit_transform(play_tennis['temp']),
    'humidity': le_humidity.fit_transform(play_tennis['humidity']),
    'wind': play_tennis['wind'].astype(int)
})

y = le_play.fit_transform(play_tennis['play'])

# Train CategoricalNB
gnb = CategoricalNB()
gnb.fit(X, y)

# Test prediction for: Sunny, Cool, High, TRUE
X_test = np.array([[
    le_outlook.transform(['sunny'])[0],
    le_temp.transform(['cool'])[0],
    le_humidity.transform(['high'])[0],
    1  # TRUE
]])

prediction = gnb.predict(X_test)
print(f"sklearn prediction: {le_play.inverse_transform(prediction)[0]}")
print(f"Class probabilities: {gnb.predict_proba(X_test)[0]}")

sklearn prediction: no
Class probabilities: [0.63454143 0.36545857]


c:\Users\pawan\anaconda34\envs\dataAnalysisPython\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but CategoricalNB was fitted with feature names
  warnings.warn(
c:\Users\pawan\anaconda34\envs\dataAnalysisPython\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but CategoricalNB was fitted with feature names
  warnings.warn(
